# OTOCs of the memory under the ATS pump (poster parameters)

The out-of-time-order correlator `C(t) = -<[x_a(t), p_a(0)]^2>` of a
two-photon-pumped ATS (memory `a`, buffer `b`), in the cat `|i alpha> +
|-i alpha>` (buffer in vacuum). `x_a(t)` is the Heisenberg-picture position,
obtained with `dq.mesolve(-H, [identity], x_a, tsave)`: an identity jump
operator contributes nothing, so this is unitary evolution of the operator.
The growth rate is a linear fit of `log C` against `kappa_b t`.

The circuit is the poster one (`w_a = 25.34 rad/ns`, `w_b = 2 w_a`, `phi_b =
0.204`, `E_J = 1.5 x 37.12 x 2pi`, `kappa = pi / 10.4`).

This notebook used to run on `chuc`, on four local GPUs (`jax.pmap` for the
heatmap), with a `Plot_setting` module from a home directory. It now runs on the
anb-compute Ray cluster. Each simulation (`otoc_point`) is one GPU task, which
returns the OTOC curve `C(t)` as plain numpy. The fits and the plots happen
here. The original is kept in `legacy_chuc/`.

## Running it

1. **Kernel**: the workspace pixi environment,
   `/workspace/.pixi/envs/default/bin/python` (**Python 3.13.x**, `ray 2.56.0`).
   These must match the cluster.
2. **Vault session**, once (lasts about a week):

   ```bash
   export VAULT_ADDR=https://vault.int.alice-bob.com
   vault login -method=oidc role=research_core_user
   ```

3. Run all cells. The first `acr.run` of a session may open an interactive
   Microsoft device-code login for GCS read access. That is expected, and the
   login is cached.

If you have already run something that imports `ray` in this kernel, restart it
first. Ray caches `RAY_AUTH_MODE` at first import.

Nothing needs shipping: the GPU image's `prod-gpu` environment already has
jax (CUDA), dynamiqs 0.3.6 and scipy, and the functions below travel by value.
The simulation code itself is unchanged: its `mesolve` call takes no
options, so dynamiqs 0.3.6's new keyword-argument style does not affect it.


In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

In [ ]:
import numpy as np

# --- constants (computed once) ---
ħ = 1
e = 1
phi_quantum = 1/2

w_a = 25.338776456203686 #GHz
w_b = 2*w_a


phi_a = 0.11
Z_a = 2 * phi_quantum**2 * phi_a**2
C_a_inv = Z_a * w_a
E_Ca = C_a_inv / 2

phi_b = 0.204
Z_b = 2 * phi_quantum**2 * phi_b**2
C_b_inv = Z_b * w_b
E_Cb = C_b_inv / 2

E_J = 37.12 * 2 * np.pi * 1.5
kappa = np.pi/10.4

# dynamiqs' default, i.e. what the original ran with. "double" for complex128
# throughout.
PRECISION = "single"


### One OTOC curve, on one GPU worker

`otoc_point` builds one of the original's two models and returns `C(t)` on
`tsave`. The full `x_a(t)` (`n_time` dense operators) stays on the worker.
jax and dynamiqs are imported inside, on the worker.

- `model="regime"` is the code of the two regime sections (`Chaotic regime`,
  `Stabilized regime`): the constants of the cell above, `n_b = (n_a + 1) // 2`,
  and the cat amplitude `sqrt(epsilon_d / (2 g2)) = sqrt(ratio / 2)` (`sqrt(5)`
  at the `ratio = 10` both sections use).
- `model="heatmap"` is the original `prepare_simulation` + `single_fit`, with
  `prepare_simulation`'s own defaults, unchanged. They are the same circuit as
  the constants above.

Both have `w_b = 2 w_a`, so the pump is static (`w_p = 2 w_a - w_b = 0`) and
`g2 = sin(epsilon_0) E_J phi_a^2 phi_b`. `J_1(epsilon_0)` in place of
`sin(epsilon_0)` holds only for `w_p != 0`. It was used here until 2026-10-12,
and underestimated `g2` by about 2x. The cat is normalized (`dq.unit`).

In [ ]:
def otoc_point(*, label, model, epsilon_0, n_a, n_b, T, n_time, ratio=10.0,
               g2=None, precision="single"):
    """C(t) = -<[x_a(t), p_a(0)]^2> on tsave, for one point, on one GPU worker."""
    import time

    import jax
    import jax.numpy as jnp
    import numpy as np

    import dynamiqs as dq

    t_start = time.time()
    dq.set_precision(precision)
    tsave = jnp.linspace(0, T, n_time)

    if model == "regime":
        # Static pump (w_p = 2 w_a - w_b = 0): sin(epsilon_0), not J_1(epsilon_0).
        g2 = np.sin(epsilon_0) * E_J * phi_a**2 * phi_b
        epsilon_d = ratio * g2

        a = dq.destroy(n_a)
        b = dq.destroy(n_b)

        varphi_a_op_tot = dq.tensor(jnp.sqrt(Z_a/2)*(a + a.dag()) / phi_quantum, dq.eye(n_b))
        varphi_b_op_tot = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b/2)*(b + b.dag()) / phi_quantum)

        alpha = jnp.sqrt(epsilon_d / (2*g2))
        cat = dq.unit(dq.coherent(n_a, 1j*alpha) + dq.coherent(n_a, 1j*-alpha))
        psi_0 = dq.tensor(cat, dq.fock(n_b, 0))

        w_d = w_b
        drive = lambda t: epsilon_d * jnp.cos(w_d * t)
        H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

        w_p=2*w_a-w_d
        pump = lambda t: - 2 * E_J * jnp.sin(epsilon_0 * jnp.cos(w_p * t))
        H_pump = dq.modulated(pump, dq.sinm(varphi_a_op_tot + varphi_b_op_tot) - varphi_a_op_tot - varphi_b_op_tot)

        H = dq.tensor(w_a * dq.number(n_a), dq.eye(n_b)) \
            + dq.tensor(dq.eye(n_a), w_b * dq.number(n_b)) \
            + H_drive + H_pump

        jumps = [dq.tensor(dq.eye(n_a), dq.eye(n_b))]

        x_a = ((dq.tensor(a, dq.eye(n_b))+dq.tensor(a, dq.eye(n_b)).dag())/jnp.sqrt(2)).asdense()
        p_0 = 1j*((dq.tensor(a, dq.eye(n_b)).dag()-dq.tensor(a, dq.eye(n_b)))/jnp.sqrt(2)).asdense()

    elif model == "heatmap":
        def prepare_simulation(n_a, n_b, circuit_params=None, init_mode_a=None, cat_phase=0.0):
            defaults = dict(
                w_a = 25.338776456203686,
                w_b = 2 * 25.338776456203686,
                E_J = 37.12 * 2 * jnp.pi * 1.5,
                epsilon_0 = 0.12,
                kappa = jnp.pi/10.4,
                epsilon_d = 0.3369,
                phi_a = 0.11,
                phi_b = 0.204,
            )

            circuit_params = {**defaults, **(circuit_params or {})}
            phi_quantum=1/2

            Z_a = 2 * phi_quantum**2 * circuit_params["phi_a"]**2
            Z_b = 2 * phi_quantum**2 * circuit_params["phi_b"]**2

            if init_mode_a is None:
                # static pump (w_b = 2 w_a): sin(epsilon_0), not J_1(epsilon_0)
                g2 = np.sin(circuit_params["epsilon_0"]) * circuit_params["E_J"] * circuit_params["phi_a"]**2 * circuit_params["phi_b"]
                alpha = jnp.sqrt(circuit_params["epsilon_d"] / (2*g2))
                init_mode_a = dq.unit(dq.coherent(n_a, 1j*alpha) + dq.coherent(n_a, 1j*-alpha))

            a, b = dq.destroy(n_a), dq.destroy(n_b)

            varphi_a = dq.tensor(jnp.sqrt(Z_a / 2) * (a + a.dag()) / phi_quantum, dq.eye(n_b))
            varphi_b = dq.tensor(dq.eye(n_a), jnp.sqrt(Z_b / 2) * (b + b.dag()) / phi_quantum)

            psi_0 = dq.tensor(init_mode_a, dq.fock(n_b, 0))

            w_d = circuit_params["w_b"]
            drive = lambda t: circuit_params["epsilon_d"] * jnp.cos(w_d * t)
            H_drive = dq.modulated(drive, dq.tensor(dq.eye(n_a), b + b.dag()))

            w_p = 2 * circuit_params["w_a"] - w_d
            pump = lambda t: -2 * circuit_params["E_J"] * jnp.sin(circuit_params["epsilon_0"] * jnp.cos(w_p * t))
            H_pump = dq.modulated(pump, dq.sinm(varphi_a + varphi_b) - varphi_a - varphi_b)

            H = (
                dq.tensor(circuit_params["w_a"] * dq.number(n_a), dq.eye(n_b))
                + dq.tensor(dq.eye(n_a), circuit_params["w_b"] * dq.number(n_b))
                + H_drive + H_pump
            )

            jumps = [dq.tensor(dq.eye(n_a), dq.eye(n_b))]

            x_a = ((dq.tensor(a, dq.eye(n_b))+dq.tensor(a, dq.eye(n_b)).dag())/jnp.sqrt(2)).asdense()
            p_a = 1j*((dq.tensor(a, dq.eye(n_b)).dag()-dq.tensor(a, dq.eye(n_b)))/jnp.sqrt(2)).asdense()

            return H, jumps, psi_0, x_a, p_a

        # single_fit, up to the fit (done on the client)
        epsilon_d = ratio * g2

        alpha = jnp.sqrt(ratio / 2)
        init_mode_a = dq.unit(dq.coherent(n_a, 1j*alpha) + dq.coherent(n_a, 1j*-alpha))
        H, jumps, psi_0, x_a, p_0 = prepare_simulation(
            n_a, n_b, init_mode_a=init_mode_a,
            circuit_params={"epsilon_0": epsilon_0, "epsilon_d": epsilon_d}
        )
    else:
        raise ValueError(f"unknown model {model!r}")

    # Heisenberg picture: mesolve with -H evolves x_a forward as an operator.
    result = dq.mesolve(-H, jumps, x_a, tsave)

    otoc = np.array([
        -np.real(dq.expect((x @ p_0 - p_0 @ x) @ (x @ p_0 - p_0 @ x), psi_0)).item()
        for x in result.states
    ])

    return {
        "label": label,
        "model": model,
        "epsilon_0": float(epsilon_0),
        "ratio": float(ratio),
        "g2": float(g2),
        "epsilon_d": float(epsilon_d),
        "n_a": n_a,
        "n_b": n_b,
        "tsave": np.asarray(tsave),
        "otoc": otoc,
        "device": str(jax.devices()[0]),
        "elapsed_s": time.time() - t_start,
    }

### The coordinator

One job per section, one GPU task per point. The coordinator only fans out and
gathers, so it is submitted with `entrypoint_num_cpus=0`, which keeps it on the
head node rather than on a freshly started spot worker (see
`/workspace/ray_driver_placement.md`). If that ever misbehaves, drop the
argument: the job then runs exactly as `acr.run`'s default.

Each point is caught on its own: a task lost to the infrastructure (node
preempted, OOM) comes back as an `"error"` entry instead of sinking the rest.

In [ ]:
import time

# Per GPU task. One whole GPU each: jax preallocates most of the card's memory.
TASK_RESOURCES = dict(num_gpus=1, num_cpus=2)


def sweep(worker, points, common, resources):
    """Coordinator: fans one GPU task out per point, gathers as they land.

    `worker` is passed in because it is defined in this notebook and ships by
    value. Results come back in the order of `points`.
    """
    import ray

    from anb_compute import ray as acr

    refs = {acr.submit(worker, **resources, **common, **p): i
            for i, p in enumerate(points)}
    out = [None] * len(points)
    pending = list(refs)
    while pending:
        ready, pending = ray.wait(pending, num_returns=1)
        i = refs[ready[0]]
        try:
            res = ray.get(ready[0])
        except Exception as exc:  # a task lost to infra, not to numerics
            res = {**points[i], "error": f"{type(exc).__name__}: {exc}"}
        out[i] = res
        print(f"{res['label']:<28} "
              + (res["error"] if "error" in res else f"{res['elapsed_s']:.1f} s"))
    return out


def run_on_cluster(worker, points, common):
    t0 = time.time()
    results = acr.run(
        sweep, worker, points, common, TASK_RESOURCES,
        poll_seconds=5,
        entrypoint_num_cpus=0,  # coordinator on the head, never on a spot node
    )
    print(f"\nwall clock, incl. job startup: {time.time() - t0:.1f} s "
          f"for {len(results)} point(s)")
    for r in results:
        if "error" in r:
            print(f"FAILED {r['label']}: {r['error']}")
    return results

In [ ]:
import matplotlib.pyplot as plt


def fit_log_otoc(res):
    """Linear fit of log C(t) against kappa_b t: slope m and R^2, as in the original."""
    t_kappa = res["tsave"] * kappa
    log_otoc = np.log(res["otoc"])
    (m, _), residuals, _, _, _ = np.polyfit(t_kappa, log_otoc, 1, full=True)
    SS_tot = sum((log_otoc - np.mean(log_otoc))**2)
    R_sq = 1 - residuals / SS_tot
    return m, R_sq.item()

# Chaotic and stabilized regimes

`epsilon_0 = 0.93` (chaotic) and `epsilon_0 = 0.2` (stabilized), at
`epsilon_d = 10 g2`, `n_a = 38`, `T = 0.2 / kappa`, in one job. The original
found `m = 29.66` (`R^2 = 0.692`) and `m = 2.468` (`R^2 = 0.0196`). Both curves
are saved, as in the original (`otoc_example_<date>.json`,
`otoc_example_stable_<date>.json`).

In [ ]:
# time discretization: reduce points if possible
T = .2 / kappa
n_time = 50   # 10000 -> 2000 for speed; increase if needed
n_a = 38  # Example value for n_a
n_b = (n_a + 1) // 2

REGIMES = [
    dict(label="chaotic, eps_0 = 0.93", epsilon_0=0.93),
    dict(label="stabilized, eps_0 = 0.2", epsilon_0=0.2),
]
COMMON = dict(model="regime", ratio=10.0, n_a=n_a, n_b=n_b, T=T, n_time=n_time,
              precision=PRECISION)

regimes = run_on_cluster(otoc_point, REGIMES, COMMON)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.4, 4.0), constrained_layout=True)
for ax, res in zip(axes, regimes):
    if "error" in res:
        ax.set_title(f"{res['label']}: FAILED", fontsize=9)
        continue
    m, R_sq = fit_log_otoc(res)
    ax.plot(res["tsave"] * kappa, res["otoc"])
    ax.set_yscale('log')
    ax.set_ylabel(r'$\langle[x_a(t),p_a(0)]^2\rangle$')
    ax.set_xlabel(r'$\kappa_b t$')
    ax.set_title(f"OTOC, {res['label']}: " + r'$n_a=$' + str(res['n_a']), fontsize=9)
    print(f"{res['label']:<28} m = {m:.6g}   R^2 = {R_sq:.6g}")
plt.show()

In [ ]:
import datetime
import json
import pathlib

# Written next to this notebook (or the workspace root, if that is the kernel's
# working directory) under results/.
here = pathlib.Path.cwd()
out_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"
out_dir.mkdir(parents=True, exist_ok=True)
date = f"{datetime.date.today():%d_%m_%Y}"

# Same format as the original: the list of C(t) values on tsave.
for name, res in (("otoc_example", regimes[0]), ("otoc_example_stable", regimes[1])):
    if "error" in res:
        print(f"{res['label']} failed, not saved")
        continue
    out_path = out_dir / f"{name}_{date}.json"
    with open(out_path, "w") as f:
        json.dump(res["otoc"].tolist(), f)
    print(f"wrote {out_path}")

# Heatmap

The fitted rate over `epsilon_0` and `epsilon_d / g2 = 2, ..., 9`, at
`n_a = 29`, `n_b = 17`, `T = 0.2 / kappa`: one GPU task per point, 184 in all.
Ray queues whatever does not fit on the GPU pool at once. (The original
batched all of them with `pmap` x `vmap` and ran out of GPU memory, asking for
9.4 GB per card. One point per task avoids that.) As in the original,
`epsilon_0` is cast to float32 before `g2` is computed, and the plotted rate is
`lyap = m / 2`. `end_otoc_arr` is `C(T)`, the last point of each curve.

`lyap_arr[i, j]`, `R_sq_arr[i, j]` and `end_otoc_arr[i, j]` are at `epsilon_0_list[i]`,
`ratio_list[j]`, the same layout as the original's arrays. A failed point is
`nan`. The fit now runs here in float64 numpy rather than in float32 jax on the
GPU, so the last digits can differ from the original.

In [ ]:
# Precompute with NumPy/Scipy
epsilon_0_list = list(np.concatenate((np.arange(0.2, .6, 0.07), np.arange(0.6, 1.1, 0.03))))
epsilon_0_arr = np.array(epsilon_0_list, dtype=np.float32)
g2_arr = np.sin(epsilon_0_arr) * E_J * phi_a**2 * phi_b   # static pump: sin, not J_1

ratio_list = np.arange(2, 10, 1)

T = .2 / kappa
n_time = 50   # 10000 -> 2000 for speed; increase if needed
n_a = 29
n_b = 17

POINTS = [dict(label=f"eps_0 = {eps:.3f}, ed/g2 = {ed:g}", epsilon_0=float(eps),
               g2=float(g2), ratio=float(ed))
          for eps, g2 in zip(epsilon_0_arr, g2_arr) for ed in ratio_list]
COMMON = dict(model="heatmap", n_a=n_a, n_b=n_b, T=T, n_time=n_time, precision=PRECISION)

heatmap = run_on_cluster(otoc_point, POINTS, COMMON)

fits = np.array([(np.nan, np.nan) if "error" in r else fit_log_otoc(r) for r in heatmap])
lyap_arr = (fits[:, 0] / 2).reshape(len(epsilon_0_arr), len(ratio_list))
R_sq_arr = fits[:, 1].reshape(len(epsilon_0_arr), len(ratio_list))
end_otoc_arr = np.array([np.nan if "error" in r else r["otoc"][-1] for r in heatmap]).reshape(
    len(epsilon_0_arr), len(ratio_list))
print(f"{np.isnan(lyap_arr).sum()} failed point(s) out of {lyap_arr.size}")

In [ ]:
print("g2 / 2pi (MHz):", np.round(g2_arr / 2 / np.pi * 1e3, 3))

In [ ]:
d_eps = (epsilon_0_arr[1] - epsilon_0_arr[0]) / 2
extent = (epsilon_0_arr[0] - d_eps, epsilon_0_arr[-1] + d_eps,
          ratio_list[0] - 0.5, ratio_list[-1] + 0.5)

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.4), constrained_layout=True)
for ax, arr, name in ((axes[0], lyap_arr, r"$m/2$"), (axes[1], R_sq_arr, r"$R^2$"),
                      (axes[2], end_otoc_arr, r"$C(T)$")):
    im = ax.imshow(arr.T, origin="lower", aspect="auto", extent=extent)
    ax.set_xlabel(r'$\varepsilon_p$' + ' (rad)')
    ax.set_ylabel(r'$\varepsilon_d / g_2$')
    ax.set_title(name + r', $n_a=$' + str(n_a) + ', ' + r'$n_b=$' + str(n_b))
    fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
import datetime
import json
import pathlib

# Same format as the original's lyap_array_, R_sq_array_ and end_otoc_array_<date>.json:
# a list of rows, one per epsilon_0, one entry per epsilon_d / g2. Written next to
# this notebook (or the workspace root, if that is the kernel's working
# directory) under results/.
here = pathlib.Path.cwd()
out_dir = (here if here.name == "Quantum_chaos" else here / "Quantum_chaos") / "results"
out_dir.mkdir(parents=True, exist_ok=True)
date = f"{datetime.date.today():%d_%m_%Y}"

for name, arr in (("lyap_array", lyap_arr), ("R_sq_array", R_sq_arr),
                  ("end_otoc_array", end_otoc_arr)):
    out_path = out_dir / f"{name}_{date}.json"
    with open(out_path, "w") as f:
        json.dump(arr.astype(float).tolist(), f)
    print(f"wrote {out_path}")